In [1]:
import pandas as pd
import numpy as np
import dask.dataframe as dd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import scipy as sc
from scipy.integrate import quad
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from scipy.signal import find_peaks
from scipy.stats import gaussian_kde
from scipy.stats import gmean, linregress, norm, beta, uniform, lognorm, pearsonr, spearmanr
from scipy.integrate import simpson, cumulative_trapezoid, trapezoid
from scipy.optimize import minimize
from scipy.special import logsumexp
from scipy.integrate import simpson
from scipy import stats
import glob
import os
import math
import re
import csv
from tqdm import tqdm
import time
import joblib
from joblib import Parallel, delayed, parallel_backend
import multiprocessing
import pickle
import random
import ast
import json
import pickle
import Generate_missense

In [2]:
# Scores except Misfit, PrimateAI-3d are here
with open(f"/n/scratch/users/p/prk534/Missense/new_scores.pkl", "rb") as f:
    df_scores_fil = pickle.load(f)

print("Done")

df_scores_fil = df_scores_fil[~(df_scores_fil["#CHROM"].isin(["X", "Y"]))]
df_scores_fil = df_scores_fil.drop(columns = ['locus', 'alleles', 'uniprot_base', 'mane_enst', 'canon_enst', 'all_enst', 'mane_NM', 
                                            'mane_select', 'canonical', 'EVE', 'esm1b_neg', 'MisFit_S', 'MisFit_D'])


Done


In [3]:
df_scores = df_scores_fil[df_scores_fil["ensg"].isin(["ENSG00000010438", "ENSG00000071054",
                                                      "ENSG00000104904", "ENSG00000122644",
                                                      "ENSG00000150991", "ENSG00000157654", "ENSG00000180881"])]

In [4]:
df_scores = df_scores.rename(columns={"ensg":"gene_id", "revel": "REVEL", "AM": "AlphaMissense", "mpc": "MPC"})
len(df_scores)

32375

In [5]:
# Get allele counts
folder = "../Data/missense/genes_bad_coverage_AC/"

cols_to_load = ["Chromosome", "Position", "Reference", "Alternate", "Filters - exomes", "VEP Annotation", "Allele Count", "Allele Number"]

dfs = []

for filepath in glob.glob(os.path.join(folder, "*.xlsx")):
    filename = os.path.basename(filepath)
    stem = os.path.splitext(filename)[0]

    # split only on the first underscore so gene_name can itself contain underscores
    gene_id, gene_name = stem.split("_", 1)

    df = pd.read_excel(filepath, usecols=cols_to_load)

    df["gene_id"] = gene_id
    df["gene_name"] = gene_name

    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)

df_all.head()

,Chromosome,Position,Reference,Alternate,Filters - exomes,VEP Annotation,Allele Count,Allele Number,gene_id,gene_name
0,9,33750834,A,T,PASS,start_lost,3,1389356,ENSG00000010438,PRSS3
1,9,33750834,A,G,PASS,start_lost,50,1389238,ENSG00000010438,PRSS3
2,9,33750835,T,C,PASS,start_lost,1,1384538,ENSG00000010438,PRSS3
3,9,33750836,G,A,PASS,start_lost,1,1391066,ENSG00000010438,PRSS3
4,9,33750840,C,A,PASS,missense_variant,1,1386014,ENSG00000010438,PRSS3


In [6]:
# 1. Rename columns as initiated in your snippet
df_all = df_all.rename(columns={"Chromosome": "#CHROM", "Position": "POS", "Reference": "REF", "Alternate": "ALT", 
                                "Filters - exomes": "FILTER", "VEP Annotation": "most_severe_consequence2",
                                "Allele Count": "allele_count", "Allele Number": "allele_number"})

# 2. Apply the filters
# Filter on allele number
# Filter on missense variants only
# Filter on "PASS" for Filters-exomes
df_all_filtered = df_all[
    (df_all["allele_number"] >= 730947 * 0.9 * 2) &             # 90% coverage
    (df_all["most_severe_consequence2"] == "missense_variant") & # Missense only
    (df_all["FILTER"] == "PASS")                                # Filters-exomes PASS
].copy()

# Preview the results
print(f"Original shape: {df_all.shape}")
print(f"Filtered shape: {df_all_filtered.shape}")
df_all_filtered.head()

Original shape: (5358, 10)
Filtered shape: (4683, 10)


,#CHROM,POS,REF,ALT,FILTER,most_severe_consequence2,allele_count,allele_number,gene_id,gene_name
4,9,33750840,C,A,PASS,missense_variant,1,1386014,ENSG00000010438,PRSS3
5,9,33750840,C,T,PASS,missense_variant,2,1386014,ENSG00000010438,PRSS3
6,9,33750844,C,A,PASS,missense_variant,2,1367438,ENSG00000010438,PRSS3
7,9,33750844,C,T,PASS,missense_variant,2,1367436,ENSG00000010438,PRSS3
8,9,33750847,G,A,PASS,missense_variant,14,1381110,ENSG00000010438,PRSS3


In [7]:
# Merge the scores with allele count
df_score_AC = df_scores.merge(df_all_filtered[["#CHROM", "POS", "REF", "ALT", "gene_id", "allele_count", "allele_number"]],
                                               on = ["#CHROM", "POS", "REF", "ALT", "gene_id"], how = "left")

In [8]:
df_score_AC['allele_count'] = df_score_AC['allele_count'].fillna(0)
# Optional: verify the change
df_score_AC.head()

,gene_id,proteinmpnn_llr_neg,REVEL,rasp_score,AlphaMissense,polyphen_score,cpt1_score,popEVE_neg,ESM_1v_neg,MPC,cadd_score,gpn_msa_score,#CHROM,POS,REF,ALT,allele_count,allele_number
0,ENSG00000071054,1.47200,0.193,-0.292250,0.4952,0.060,0.24643,4.921,5.046,1.8236,4.2969,-9.82,2,101698084,G,A,0.0,NaN
1,ENSG00000071054,2.04980,0.312,0.296230,0.5796,0.169,0.30046,4.930,4.591,2.3998,4.2898,-10.46,2,101698084,G,C,0.0,NaN
2,ENSG00000071054,0.79921,0.103,0.098353,0.1918,0.021,0.16918,4.418,3.211,1.3339,4.1019,-10.31,2,101698084,G,T,1.0,1327774.0
3,ENSG00000071054,0.97259,0.193,-0.281340,0.8041,0.001,0.45727,5.084,7.893,2.3380,3.7190,-9.58,2,101698085,C,A,1.0,1325930.0
4,ENSG00000071054,1.56610,0.071,0.781690,0.2925,0.072,0.27865,4.742,5.961,1.9514,4.1146,-10.47,2,101698085,C,G,1.0,1325930.0


In [12]:
parent_dir = "/n/data2/hms/dbmi/sunyaev/lab/dlee/kl/data/whole_genome/mu_filtered/"

# 1. Identify the unique chromosomes actually present in your 7 genes
# Ensuring they are strings to match the folder naming convention
target_chroms = df_score_AC['#CHROM'].astype(str).unique()

all_mr_subsets = []

for chrom in target_chroms:
    folder_path = os.path.join(parent_dir, chrom)
    
    if os.path.exists(folder_path):
        print(f"Processing Chromosome {chrom}...")
        
        # 2. Read only 'Pos', 'mu', 'Allele_ref', 'Allele' from the parquet files
        # We use .compute() to bring the small subset into a pandas dataframe
        df_mr_chrom = dd.read_parquet(folder_path, columns=['Pos', 'mu', 'Allele_ref', 'Allele']).compute()
        
        # 3. Rename for a clean merge
        df_mr_chrom = df_mr_chrom.rename(columns={'Pos': 'POS', 'mu': 'MR', 'Allele_ref': 'REF', 'Allele': 'ALT'})
        
        # 4. Filter MR data to only include POS present in your df_score_AC for this chrom
        # This keeps the merge operation very fast and memory-light
        df_subset = df_score_AC[df_score_AC['#CHROM'].astype(str) == chrom]
        merged_subset = df_subset.merge(df_mr_chrom, on=['POS', 'REF', 'ALT'], how='left')
        
        all_mr_subsets.append(merged_subset)
    else:
        print(f"Warning: Directory for Chromosome {chrom} not found.")
        all_mr_subsets.append(df_score_AC[df_score_AC['#CHROM'].astype(str) == chrom])

# 5. Combine the processed chunks back into one dataframe
df_score_AC = pd.concat(all_mr_subsets).reset_index(drop=True)

# Preview result
df_score_AC.head()

Processing Chromosome 2...
Processing Chromosome 7...
Processing Chromosome 9...
Processing Chromosome 12...
Processing Chromosome 19...


,gene_id,proteinmpnn_llr_neg,REVEL,rasp_score,AlphaMissense,polyphen_score,cpt1_score,popEVE_neg,ESM_1v_neg,MPC,cadd_score,gpn_msa_score,#CHROM,POS,REF,ALT,allele_count,allele_number,MR
0,ENSG00000071054,1.47200,0.193,-0.292250,0.4952,0.060,0.24643,4.921,5.046,1.8236,4.2969,-9.82,2,101698084,G,A,0.0,NaN,0.139
1,ENSG00000071054,2.04980,0.312,0.296230,0.5796,0.169,0.30046,4.930,4.591,2.3998,4.2898,-10.46,2,101698084,G,C,0.0,NaN,0.030
2,ENSG00000071054,0.79921,0.103,0.098353,0.1918,0.021,0.16918,4.418,3.211,1.3339,4.1019,-10.31,2,101698084,G,T,1.0,1327774.0,0.041
3,ENSG00000071054,0.97259,0.193,-0.281340,0.8041,0.001,0.45727,5.084,7.893,2.3380,3.7190,-9.58,2,101698085,C,A,1.0,1325930.0,0.094
4,ENSG00000071054,1.56610,0.071,0.781690,0.2925,0.072,0.27865,4.742,5.961,1.9514,4.1146,-10.47,2,101698085,C,G,1.0,1325930.0,0.083


In [17]:
# Check MR coverage per gene
gene_check = df_score_AC.groupby('gene_id').agg(
    total_variants=('MR', 'size'),
    missing_MR=('MR', lambda x: x.isna().sum()),
    percent_missing=('MR', lambda x: (x.isna().sum() / len(x)) * 100)
)

print(gene_check)

# Specifically check if any gene has 100% missing MR
completely_missing = gene_check[gene_check['percent_missing'] == 100]
if not completely_missing.empty:
    print("\nWarning: The following genes are completely missing MR values:")
    print(completely_missing.index.tolist())
else:
    print("\nSuccess: Every gene has at least some MR coverage.")

                 total_variants  missing_MR  percent_missing
gene_id                                                     
ENSG00000010438            1812        1780        98.233996
ENSG00000071054            9131         150         1.642755
ENSG00000104904            2041          64         3.135718
ENSG00000122644            1297        1297       100.000000
ENSG00000150991            4519        4519       100.000000
ENSG00000157654            9253          13         0.140495
ENSG00000180881            4322         182         4.211013

['ENSG00000122644', 'ENSG00000150991']


In [23]:
# Create the QUAL column
# If MR is not NaN, label as 'high/TFBS', otherwise label as NaN (or another placeholder)
df_score_AC['QUAL'] = df_score_AC['MR'].apply(lambda x: 'high/TFBS' if pd.notna(x) else np.nan)

# Verify the assignment for the genes with good coverage
print(df_score_AC.groupby(['gene_id', 'QUAL']).size())

gene_id          QUAL     
ENSG00000010438  high/TFBS      32
ENSG00000071054  high/TFBS    8981
ENSG00000104904  high/TFBS    1977
ENSG00000157654  high/TFBS    9240
ENSG00000180881  high/TFBS    4140
dtype: int64


In [27]:
# 1. Define the genes that need help
bad_genes = ['ENSG00000122644', 'ENSG00000150991', 'ENSG00000010438']

# 2. Iterate only through chromosomes where these genes exist
target_chroms = df_score_AC.loc[df_score_AC['gene_id'].isin(bad_genes), '#CHROM'].unique()

for chrom in target_chroms:
    # Ensure chrom is a string to match file path
    chrom_str = str(chrom)
    supp_path = f"/n/scratch/users/p/prk534/{chrom_str}_rate_v5.2_TFBS_correction_all.vcf.gz"
    
    print(f"Supplementing Chromosome {chrom_str}...")
    
    # Read the VCF
    vcf_cols = ['#CHROM', 'POS', 'ID', 'REF', 'ALT', 'QUAL_VCF', 'FILTER', 'INFO']
    df_supp = pd.read_csv(supp_path, sep="\t", comment='#', names=vcf_cols, 
                          usecols=['POS', 'REF', 'ALT', 'FILTER', 'INFO'])

    # Extract MR from INFO
    df_supp['MR_supp'] = df_supp['INFO'].str.extract(r'MR=([^;]+)').astype(float)
    
    # 3. CRITICAL FIX: The mask must include the CHROMOSOME condition
    # This prevents overwriting genes on other chromosomes with NaNs
    rows_to_fill = (df_score_AC['gene_id'].isin(bad_genes)) & \
                   (df_score_AC['MR'].isna()) & \
                   (df_score_AC['#CHROM'].astype(str) == chrom_str)
    
    if not rows_to_fill.any():
        continue

    # 4. Perform the merge only for this chromosome's missing rows
    subset = df_score_AC[rows_to_fill].drop(columns=['MR', 'QUAL'])
    merged_subset = subset.merge(df_supp[['POS', 'REF', 'ALT', 'MR_supp', 'FILTER']], 
                                 on=['POS', 'REF', 'ALT'], 
                                 how='left')
    
    # 5. Map the new values back using the index of the filtered rows
    # We align the index of merged_subset back to the original dataframe
    merged_subset.index = df_score_AC[rows_to_fill].index
    
    df_score_AC.loc[rows_to_fill, 'MR'] = merged_subset['MR_supp']
    df_score_AC.loc[rows_to_fill, 'QUAL'] = merged_subset['FILTER']

# Final check
print(df_score_AC.groupby('gene_id')['MR'].isna().sum())

Supplementing Chromosome 7...
Supplementing Chromosome 9...
Supplementing Chromosome 12...


AttributeError: 'SeriesGroupBy' object has no attribute 'isna'

In [28]:
print(df_score_AC.groupby('gene_id')['MR'].apply(lambda x: x.isna().sum()))

gene_id
ENSG00000010438      0
ENSG00000071054    150
ENSG00000104904     64
ENSG00000122644      0
ENSG00000150991      0
ENSG00000157654     13
ENSG00000180881    182
Name: MR, dtype: int64


In [29]:
# Create a copy or overwrite to remove rows where MR is still NaN
df_score_AC = df_score_AC.dropna(subset=['MR']).reset_index(drop=True)

In [33]:
# # Save the dataframe to a compressed .txt.gz file
# df_score_AC.to_csv("../Data/missense/genes_bad_coverage_AC/score_AC_missing_genes.txt.gz", sep="\t", index=False, compression="gzip")

# print("File saved successfully as score_AC_missing_genes.txt.gz")

File saved successfully as score_AC_missing_genes.txt.gz


In [2]:
df_score_AC = pd.read_csv("../Data/../Data/missense/genes_bad_coverage_AC/score_AC_missing_genes.txt.gz", sep = "\t", compression="gzip")

In [3]:
len(df_score_AC)

31966

In [4]:
# Define the exclusion masks for each gene
# We use ~ to negate the condition (i.e., "keep if NOT in these ranges")

# 1. UBC (ENSG00000150991)
mask_ubc = (df_score_AC['gene_id'] == 'ENSG00000150991') & \
           (df_score_AC['POS'].between(124911935, 124912554))

# 2. PRSS3 (ENSG00000010438)
mask_prss3 = (df_score_AC['gene_id'] == 'ENSG00000010438') & \
             (df_score_AC['POS'] < 33794742)

# 3. OAZ1 (ENSG00000104904)
mask_oaz1 = (df_score_AC['gene_id'] == 'ENSG00000104904') & \
            (df_score_AC['POS'].between(2270000, 2270900))

# 4. PALM2AKAP2 (ENSG00000157654)
mask_palm = (df_score_AC['gene_id'] == 'ENSG00000157654') & \
            (df_score_AC['POS'].between(109866000, 109867200) | 
             df_score_AC['POS'].between(110090000, 110091000) |
             (df_score_AC['POS'] <109640900))

# 5. MAP4K4 (ENSG00000071054)
mask_map4k4 = (df_score_AC['gene_id'] == 'ENSG00000071054') & \
              (df_score_AC['POS'].between(101698000, 101698200) | 
               df_score_AC['POS'].between(101863700, 101864100))

# 6. CAPS2 (ENSG00000180881)
mask_caps2 = (df_score_AC['gene_id'] == 'ENSG00000180881') & \
             (df_score_AC['POS'].between(75282200, 75282400) | 
              df_score_AC['POS'].between(75291700, 75291900) | 
              df_score_AC['POS'].between(75299800, 75300000) | 
              df_score_AC['POS'].between(75312800, 75313000) | 
              df_score_AC['POS'].between(75321300, 75321600) | 
              (df_score_AC['POS'] > 75326300))

# Combine all exclusion masks
exclude_mask = mask_ubc | mask_prss3 | mask_oaz1 | mask_palm | mask_map4k4 | mask_caps2

# Filter the dataframe
df_excluded = df_score_AC[exclude_mask].reset_index(drop=True)
df_score_AC = df_score_AC[~exclude_mask].reset_index(drop=True)

# Verify the result
print(f"Rows remaining: {len(df_score_AC)}")
print(df_score_AC.groupby('gene_id').size())

Rows remaining: 27795
gene_id
ENSG00000010438    1779
ENSG00000071054    8356
ENSG00000104904    1499
ENSG00000122644    1297
ENSG00000150991    3154
ENSG00000157654    9161
ENSG00000180881    2549
dtype: int64


In [5]:
# Get AlphaMissense based s values for these genes

In [6]:
s_values = -np.logspace(-6, 2, num=100)
log_s_grid = np.log10(-s_values)
dx = log_s_grid[1] - log_s_grid[0]
# MLE params
output_file = "../Data/missense/genes_bad_coverage_AC/AlphaMissense_norm.pkl"
lof_ref_path = "../LoF_selection/LoF_s_het.txt.gz"
init_c= 0.1
init_beta=1.0
init_log_sigma=np.log(0.5551)
bounds = [(-4, 4), (0, 10), (np.log(dx), np.log(10.0))]
maxiter=400
fatol=1e-8
# parallel
n_jobs=-1
backend="multiprocessing"
verbose=10
# outputs
gene_mle_out_dir=None
# optional LoF dedup
lof_ref_path=None
lof_ref_score_col="Posterior_CI10_lower"
key_cols = ["#CHROM", "POS", "REF", "ALT"]

In [7]:
# 2) filter + MR rank
score_col = "AlphaMissense"
allele_count_max=5000
df = Generate_missense.filter_variants(df_score_AC, score_col, allele_count_max=allele_count_max)

In [8]:
# 1. Load the existing MR values from your file
# Assuming the column is named 'MR' based on your previous zgrep output
file_path = "../Mutation_rate_estimation/param_mut_rate/results_mu_var_p_all.txt"
existing_mr_df = pd.read_csv(file_path, sep='\t')
mr_values = existing_mr_df['MR'].unique().tolist()

# 2. Add the missing value and sort
missing_value = 0.004
if missing_value not in mr_values:
    mr_values.append(missing_value)

# Sorting in ascending order
mr_sorted = sorted(mr_values)

# 3. Create a mapping dictionary {MR_value: Index}
# This will index them from 0 to 98 (total 99 values)
mr_to_index = {val: i for i, val in enumerate(mr_sorted)}

# 4. Map the values in your main DataFrame 'df'
# Replace 'MR_column' with the actual column name in your df
df['MR_rank'] = df['MR'].map(mr_to_index)

# Verification
print(f"Total MR values: {len(mr_sorted)}")
print(f"First 5 mapped values:\n{df[['MR', 'MR_rank']].head()}")

Total MR values: 99
First 5 mapped values:
      MR  MR_rank
0  0.094        9
1  0.041        4
2  0.083        8
3  0.020        2
4  0.083        8


In [14]:
if score_col == "AlphaMissense":
    df[f"{score_col}_norm"] = df.groupby("gene_id")[score_col].transform(Generate_missense.inverse_normal_transform)
    score_col = f"{score_col}_norm"

# 3) per-gene
gene_dfs = Generate_missense.construct_gene_dfs(df)

sfs_mle_pkl="/n/data2/hms/dbmi/sunyaev/lab/pkar/demography_SFS/SFS_all_missense.pkl"
sfs_mle = Generate_missense.load_sfs(sfs_mle_pkl)
s_values_mle = -np.logspace(-6, 2, num=100)
sfs_variant_pkl="/n/data2/hms/dbmi/sunyaev/lab/pkar/demography_SFS/SFS_all.pkl"
sfs_variant = Generate_missense.load_sfs(sfs_variant_pkl)
s_values_variant = -np.logspace(-6, 2, num=500)

In [15]:
gene_mle_out_dir = os.path.join(os.path.dirname(output_file) or ".", f"gene_mle_results_{score_col}")
df_summary = Generate_missense.fit_all_genes_mle(gene_dfs, s_values_mle, sfs_mle, score_col, 
                                                 init_c=init_c, init_beta=init_beta, init_log_sigma=init_log_sigma,
                                                 bounds=bounds, maxiter=maxiter, fatol=fatol,
                                                 n_jobs=n_jobs, backend=backend, verbose=verbose,
                                                 out_dir=gene_mle_out_dir)

[Parallel(n_jobs=-1)]: Using backend MultiprocessingBackend with 19 concurrent workers.
[Parallel(n_jobs=-1)]: Done   1 tasks      | elapsed:    1.6s
[Parallel(n_jobs=-1)]: Done   2 out of   7 | elapsed:    1.7s remaining:    4.1s
[Parallel(n_jobs=-1)]: Done   3 out of   7 | elapsed:    2.1s remaining:    2.8s
[Parallel(n_jobs=-1)]: Done   4 out of   7 | elapsed:    2.2s remaining:    1.7s
[Parallel(n_jobs=-1)]: Done   5 out of   7 | elapsed:    4.2s remaining:    1.7s
[Parallel(n_jobs=-1)]: Done   7 out of   7 | elapsed:   13.1s finished


In [17]:
out_prior_col = f"s_prior_{score_col}"
out_post_col = f"s_{score_col}"
df_all = Generate_missense.attach_medians_all_genes(gene_dfs, df_summary, score_col, sfs_variant, s_values_variant, out_prior_col, out_post_col,
                                                    n_jobs=n_jobs, backend=backend, verbose=verbose)

[Parallel(n_jobs=-1)]: Using backend MultiprocessingBackend with 19 concurrent workers.
[Parallel(n_jobs=-1)]: Done   1 tasks      | elapsed:    1.0s
[Parallel(n_jobs=-1)]: Done   2 out of   7 | elapsed:    1.0s remaining:    2.6s
[Parallel(n_jobs=-1)]: Done   3 out of   7 | elapsed:    1.0s remaining:    1.4s
[Parallel(n_jobs=-1)]: Done   4 out of   7 | elapsed:    1.1s remaining:    0.8s
[Parallel(n_jobs=-1)]: Done   5 out of   7 | elapsed:    1.1s remaining:    0.4s
[Parallel(n_jobs=-1)]: Done   7 out of   7 | elapsed:    1.5s finished


In [18]:
df_all

,gene_id,proteinmpnn_llr_neg,REVEL,rasp_score,AlphaMissense,polyphen_score,cpt1_score,popEVE_neg,ESM_1v_neg,MPC,...,REF,ALT,allele_count,allele_number,MR,QUAL,MR_rank,AlphaMissense_norm,s_prior_AlphaMissense_norm,s_AlphaMissense_norm
0,ENSG00000071054,2.5986,0.324,0.61649,0.9713,0.069,0.45400,5.288,8.636,2.102200,...,G,A,0.0,NaN,0.094,high/TFBS,9,0.735847,0.108160,0.120827
1,ENSG00000071054,3.8196,0.411,0.49598,0.9928,0.503,0.72504,6.009,13.754,2.354500,...,G,C,0.0,NaN,0.041,high/TFBS,4,1.205269,0.202588,0.218111
2,ENSG00000071054,5.4035,0.472,0.45240,0.9814,0.644,0.73018,6.132,14.712,2.777000,...,G,T,0.0,NaN,0.083,high/TFBS,8,0.864803,0.130085,0.145320
3,ENSG00000071054,4.0185,0.304,1.27500,0.9902,0.033,0.75988,6.249,15.316,2.560800,...,A,C,0.0,NaN,0.020,high/TFBS,2,1.096779,0.174777,0.181350
4,ENSG00000071054,5.0741,0.274,2.51850,0.9944,0.000,0.69764,6.088,14.713,2.345400,...,A,G,0.0,NaN,0.083,high/TFBS,8,1.300954,0.234823,0.252817
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27236,ENSG00000104904,1.8685,NaN,0.11952,0.1357,0.292,0.12549,NaN,NaN,0.053343,...,A,C,0.0,NaN,0.030,high/TFBS,3,-0.694610,0.000551,0.000551
27237,ENSG00000104904,2.4877,NaN,0.58228,0.1547,0.403,0.13902,NaN,NaN,0.078681,...,A,G,0.0,NaN,0.073,high/TFBS,7,-0.569746,0.000616,0.000616
27238,ENSG00000104904,1.9831,NaN,0.25948,0.1868,0.403,0.15601,NaN,NaN,0.140250,...,A,T,0.0,NaN,0.041,high/TFBS,4,-0.396353,0.000741,0.000741
27239,ENSG00000104904,2.0193,NaN,-0.18865,0.0774,0.217,0.12192,NaN,NaN,0.008736,...,G,C,1.0,1609620.0,0.041,high/TFBS,4,-1.576228,0.000227,0.000227


In [19]:
df_all.columns

Index(['gene_id', 'proteinmpnn_llr_neg', 'REVEL', 'rasp_score',
       'AlphaMissense', 'polyphen_score', 'cpt1_score', 'popEVE_neg',
       'ESM_1v_neg', 'MPC', 'cadd_score', 'gpn_msa_score', '#CHROM', 'POS',
       'REF', 'ALT', 'allele_count', 'allele_number', 'MR', 'QUAL', 'MR_rank',
       'AlphaMissense_norm', 's_prior_AlphaMissense_norm',
       's_AlphaMissense_norm'],
      dtype='object')

In [20]:
with open(f"/n/data2/hms/dbmi/sunyaev/lab/pkar/Missense_analysis/final/all_new_scores.pkl", "rb") as f:
    df_all_variants = pickle.load(f)
print("Done")

Done


In [22]:
# Get unique gene_ids from both DataFrames
ids_all = df_all['gene_id'].unique()
ids_variants = df_all_variants['gene_id'].unique()

# Find the intersection
common_genes = np.intersect1d(ids_all, ids_variants)

print(f"common gene_ids: {common_genes}")

# Optional: To see the actual list of common IDs
# print(common_genes)

common gene_ids: []


In [23]:
# 1. Combine the dataframes row-wise
# This will align columns that exist in both and fill missing ones with NaN
df_combined = pd.concat([df_all_variants, df_all], ignore_index=True, sort=False)

# Count duplicates based on specific variant columns
duplicate_count = df_combined.duplicated(subset=['gene_id', '#CHROM', 'POS', 'REF', 'ALT']).sum()

print(f"Number of duplicate variants: {duplicate_count}")

# # 2. (Recommended) Remove duplicates to ensure genes already in df_all_variants aren't doubled
# # We use the unique variant identifiers: Chromosome, Position, Ref, Alt, and Gene ID
# df_combined = df_combined.drop_duplicates(subset=['gene_id', '#CHROM', 'POS', 'REF', 'ALT'])

# # 3. Verify the new size
# print(f"Original variants: {len(df_all_variants)}")
# print(f"New combined variants: {len(df_combined)}")

Number of duplicate variants: 0


In [24]:
df_all_variants = df_combined
df_all_variants

,gene_id,proteinmpnn_llr_neg,rasp_score,polyphen_score,cpt1_score,popEVE_neg,ESM_1v_neg,cadd_score,gpn_msa_score,#CHROM,...,PrimateAI-3D,REVEL,MR_rank,AlphaMissense_norm,s_prior_AlphaMissense_norm,s_AlphaMissense_norm,s_prior_popEVE_neg,s_popEVE_neg,s_prior_esm_score_neg,s_esm_score_neg
0,ENSG00000187634,0.98394,0.19790,0.609,0.16339,3.804,3.519,4.7263,-10.62,1,...,0.685,0.187,3.0,1.041699,0.000891,0.000891,0.000211,0.000211,0.000001,0.000001
1,ENSG00000187634,1.76220,1.30040,0.972,0.21579,3.808,3.523,5.3936,-10.65,1,...,0.773,0.273,5.0,1.437322,0.001797,0.001797,0.000211,0.000211,0.000001,0.000001
2,ENSG00000187634,1.16830,0.38739,0.545,0.15907,3.700,3.740,4.2755,-9.35,1,...,0.652,0.201,2.0,0.967232,0.000769,0.000769,0.000136,0.000136,0.000001,0.000001
3,ENSG00000187634,1.82470,-0.25759,0.910,0.26912,3.965,6.997,5.3361,-11.79,1,...,0.869,0.332,3.0,1.843850,0.003901,0.003901,0.000426,0.000426,0.000001,0.000001
4,ENSG00000187634,2.93100,0.80141,0.986,0.24496,3.954,6.095,5.4720,-9.52,1,...,0.814,0.345,7.0,1.390774,0.001669,0.001669,0.000396,0.000396,0.000001,0.000001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60283499,ENSG00000104904,1.86850,0.11952,0.292,0.12549,NaN,NaN,2.6491,-5.76,19,...,NaN,NaN,3.0,-0.694610,0.000551,0.000551,NaN,NaN,NaN,NaN
60283500,ENSG00000104904,2.48770,0.58228,0.403,0.13902,NaN,NaN,3.5168,-4.40,19,...,NaN,NaN,7.0,-0.569746,0.000616,0.000616,NaN,NaN,NaN,NaN
60283501,ENSG00000104904,1.98310,0.25948,0.403,0.15601,NaN,NaN,3.3171,-6.35,19,...,NaN,NaN,4.0,-0.396353,0.000741,0.000741,NaN,NaN,NaN,NaN
60283502,ENSG00000104904,2.01930,-0.18865,0.217,0.12192,NaN,NaN,1.9602,-4.24,19,...,NaN,NaN,4.0,-1.576228,0.000227,0.000227,NaN,NaN,NaN,NaN


In [26]:
df_all_variants["sx_prior_AlphaMissense_norm"] = (df_all_variants["s_prior_AlphaMissense_norm"] * df_all_variants["allele_count"])
df_all_variants["sx_AlphaMissense_norm"] = (df_all_variants["s_AlphaMissense_norm"] * df_all_variants["allele_count"])

In [27]:
def add_variant_significance(df, score_cols):
    """
    df: DataFrame with #CHROM, POS, and score columns.
    score_cols: list of score column names to classify.
    """

    # --- Threshold dictionary ---
    labels=("20pct", "35pct")
    
    top_percentages = {
        "20pct": 0.20,
        "35pct": 0.35
    }

    for score in score_cols:
        x = df[score]

        # mask NaN values (they remain Unknown)
        mask_finite = np.isfinite(x)

        # Benign threshold = bottom 50%
        bottom_thr = np.nanquantile(x, 0.2)

        # compute thresholds from finite values only
        for label in labels:
            top_p = top_percentages[label]

            # Top pathogenic threshold
            top_thr = np.nanquantile(x, 1 - top_p)

            colname = f"Significance_{label}_{score}"

            # initialize with Unknown
            df[colname] = "Unknown significance"

            # Benign = bottom 50%
            df.loc[(x <= bottom_thr) & mask_finite, colname] = "Benign"

            # Pathogenic = top X%
            df.loc[(x >= top_thr) & mask_finite, colname] = "Pathogenic"

            # NaNs remain "Unknown significance"
            # (no action needed)

    return df


In [28]:
df_labels = add_variant_significance(df_all_variants[["#CHROM", "POS", "REF", "ALT", "gene_id", "gene_symbol", 
                                                      "allele_count", "allele_number",
                                                      "AlphaMissense", "popEVE_neg", 
                                                      "s_prior_AlphaMissense_norm", "s_AlphaMissense_norm",
                                                      "sx_prior_AlphaMissense_norm", "sx_AlphaMissense_norm"]]
                                     , ["AlphaMissense", "popEVE_neg", "s_prior_AlphaMissense_norm", "s_AlphaMissense_norm"])

# --- Create CHROM_POS column ---
df_labels["CHROM_POS"] = df_labels["#CHROM"].astype(str) + ":" + df_labels["POS"].astype(str)
df_labels = df_labels.drop(columns=["#CHROM", "POS"])

len(df_labels)

/tmp/ipykernel_4132034/285939878.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[colname] = "Unknown significance"
/tmp/ipykernel_4132034/285939878.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[colname] = "Unknown significance"
/tmp/ipykernel_4132034/285939878.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/s

60283504

In [29]:
def add_variant_significance(df, score_cols):
    """
    df: DataFrame with score columns.
    score_cols: list of score column names to classify.

    Both benign and pathogenic thresholds are computed only from
    finite, non-zero values.
    """

    benign_percentages = {
        "10pct": 0.10,
        "20pct": 0.20
    }

    pathogenic_percentages = {
        "20pct": 0.20,
        "35pct": 0.35
    }

    for score in score_cols:
        x = df[score]

        mask_finite = np.isfinite(x)
        mask_nonzero = mask_finite & (x != 0)

        if mask_nonzero.sum() == 0:
            for b_label in benign_percentages:
                for p_label in pathogenic_percentages:
                    colname = f"Significance_path{p_label}_ben{b_label}_{score}"
                    df[colname] = "Unknown significance"
            continue

        x_nonzero = x[mask_nonzero]

        for b_label, b_pct in benign_percentages.items():
            bottom_thr = np.nanquantile(x_nonzero, b_pct)

            for p_label, p_pct in pathogenic_percentages.items():
                top_thr = np.nanquantile(x_nonzero, 1 - p_pct)

                colname = f"Significance_path{p_label}_ben{b_label}_{score}"
                df[colname] = "Unknown significance"

                # Pathogenic first
                path_mask = mask_nonzero & (x >= top_thr)

                # Benign only among remaining rows
                benign_mask = mask_nonzero & (x <= bottom_thr)

                df.loc[path_mask, colname] = "Pathogenic"
                df.loc[benign_mask, colname] = "Benign"

    return df

In [30]:
df_labels = add_variant_significance(df_labels, score_cols= ["sx_prior_AlphaMissense_norm", "sx_AlphaMissense_norm"])

In [31]:
df_syn = pd.read_csv("../Data/synonymous_data.txt.gz", sep = '\t', compression = 'gzip')
df_syn.columns

Index(['#CHROM', 'POS', 'REF', 'ALT', 'QUAL', 'MR', 'gene_id', 'transcript_id',
       'allele_count', 'allele_number', 'AN_AFR', 'AN_AMR', 'AN_EAS', 'AN_FIN',
       'AN_NFE', 'AN_SAS', 'AC_AFR', 'AC_AMR', 'AC_EAS', 'AC_FIN', 'AC_NFE',
       'AC_SAS', 'consequence', 'most_severe_consequence',
       'most_severe_consequence2', 'am', 'esm1v_neg', 'esm_neg'],
      dtype='object')

In [32]:
df_syn["CHROM_POS"] = df_syn["#CHROM"].astype(str) + ":" + df_syn["POS"].astype(str)
df_syn = df_syn.drop(columns=['#CHROM', 'POS', 'QUAL', 'MR', 'transcript_id',
                              'AN_AFR', 'AN_AMR', 'AN_EAS', 'AN_FIN','AN_NFE', 'AN_SAS', 'AC_AFR', 'AC_AMR', 'AC_EAS', 'AC_FIN', 'AC_NFE', 'AC_SAS',
                              'consequence', 'most_severe_consequence', 'most_severe_consequence2', 'am', 'esm1v_neg', 'esm_neg'])
cols_to_add = ['Significance_20pct_AlphaMissense', 'Significance_35pct_AlphaMissense', 'Significance_20pct_popEVE_neg', 'Significance_35pct_popEVE_neg', 
               'Significance_20pct_s_prior_AlphaMissense_norm', 'Significance_35pct_s_prior_AlphaMissense_norm',
               'Significance_20pct_s_AlphaMissense_norm', 'Significance_35pct_s_AlphaMissense_norm',
               'Significance_path20pct_ben10pct_sx_prior_AlphaMissense_norm',
               'Significance_path35pct_ben10pct_sx_prior_AlphaMissense_norm',
               'Significance_path20pct_ben20pct_sx_prior_AlphaMissense_norm',
               'Significance_path35pct_ben20pct_sx_prior_AlphaMissense_norm',
               'Significance_path20pct_ben10pct_sx_AlphaMissense_norm',
               'Significance_path35pct_ben10pct_sx_AlphaMissense_norm',
               'Significance_path20pct_ben20pct_sx_AlphaMissense_norm',
               'Significance_path35pct_ben20pct_sx_AlphaMissense_norm']
df_syn[cols_to_add] = "synonymous"
df_syn = df_syn.reset_index(drop = True)

In [33]:
df_combined = pd.concat([df_labels, df_syn], ignore_index=True)

In [34]:
df_combined = df_combined.drop_duplicates(subset=["CHROM_POS", "REF", "ALT"], keep=False)
df_combined = df_combined.drop(columns = ["gene_symbol"])
df_gene_names = pd.read_csv("../Data/ENS2Gene.txt", sep="\t", header=None, names=["gene_id", "gene_name"])
df_combined = df_combined.merge(df_gene_names, on = "gene_id", how = "left")
print(len(df_combined))
df_combined = df_combined[~(df_combined["gene_name"].isna())]
print(len(df_combined))

78816570
78816570


In [36]:
df_combined.to_csv("/n/data2/hms/dbmi/sunyaev/lab/pkar/Missense_analysis/final/NERINE_s_times_AC_added.txt.gz", sep="\t", index=False, compression="gzip")

In [26]:
#Scratch

In [25]:
# Add the missing MR values especially in ARL4A, UBC and PRSS3
# Peek at the first few lines of the supplementary file
# Adjust the path if the file is in a different subdirectory
supp_file_path = "/n/scratch/users/p/prk534/12_rate_v5.2_TFBS_correction_all.vcf.gz"

# This skips the '##' lines and shows the column names + first 5 data rows
!zgrep -v "^##" {supp_file_path} | head -n 6

#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO
12	10100	.	A	C	.	low	PN=TAACC;MR=0.02;MG=0.021
12	10100	.	A	G	.	low	PN=TAACC;MR=0.051;MG=0.094
12	10100	.	A	T	.	low	PN=TAACC;MR=0.02;MG=0.021
12	10101	.	C	A	.	low	PN=AACCC;MR=0.163;MG=0.104
12	10101	.	C	G	.	low	PN=AACCC;MR=0.041;MG=0.049
/usr/bin/grep: write error: Broken pipe

gzip: stdout: Broken pipe
